# NeuroGPT — the actual, published fine-tuning approach

This runs the vendored `src/train_gpt.py` completely unmodified — the paper's own downstream fine-tuning recipe (`--ft-only-encoder=True`, matching `scripts/finetune.sh`), no novelty. See the companion notebook `NeuroGPT_Novelty_PromptTuning.ipynb` for our soft prompt-tuning addition.
**Before running**: Runtime > Change runtime type > GPU (T4 is enough).

This notebook clones the original vendored repo fresh — it does not depend on any local files, so it runs standalone in Colab.

## 1. Environment

In [ ]:
!nvidia-smi

In [ ]:
!git clone https://github.com/wenhui0206/NeuroGPT.git
%cd NeuroGPT
!pip install -q -r requirements.txt

## 2. Pretrained checkpoint

Downloaded from the vendor's own HuggingFace repo, linked from their README: https://huggingface.co/wenhuic/Neuro-GPT/tree/main

In [ ]:
!mkdir -p pretrained_model
!wget -q --show-progress -O pretrained_model/pytorch_model.bin \
  "https://huggingface.co/wenhuic/Neuro-GPT/resolve/main/pretrained_model/pytorch_model.bin"
!ls -lh pretrained_model/  # expect ~303MB (318189905 bytes) -- verified against the HF repo's own API listing

## 3. Prepare BCI Competition IV 2a

`src/batcher/downstream_dataset.py::MotorImageryDataset` expects, per subject/session, an `.npz` with keys `s`, `etyp`, `epos`, `edur`, `artifacts` (the original BioSig/GDF field names), plus a `true_labels/<name>.mat` file. The vendored repo does not ship a converter for this, so this notebook builds one from the raw `.gdf` files.

**Manual step (data is behind a usage agreement, so this can't be scripted):**
1. Go to https://www.bbci.de/competition/iv/#dataset2a, agree to the terms, and download the 18 raw `.gdf` files (`A01T.gdf` ... `A09E.gdf`) plus `true_labels.zip`.
2. Upload them to your Google Drive under `neurogpt_data/raw_bci2a/` (and the unzipped label files under `neurogpt_data/raw_bci2a/true_labels/`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

RAW_DIR = '/content/drive/MyDrive/neurogpt_data/raw_bci2a'
OUT_DIR = '/content/drive/MyDrive/neurogpt_data/bci2a_egg_npz'
RESULTS_DIR = '/content/drive/MyDrive/neurogpt_results'
import os
assert os.path.isdir(RAW_DIR), f'Upload the raw BCI-IV-2a files to {RAW_DIR} first (see markdown above).'

Conversion notes (verify the sanity-check counts below before trusting a full run):
- `s`: continuous signal in microvolts, shape `(n_times, n_channels)` — `MotorImageryDataset` transposes it back internally and slices the first 22 channels (EEG only; the GDF files also carry 3 EOG channels after the 22 EEG channels, in that order).
- `etyp`/`epos`/`edur`: GDF annotation `description` (the original numeric BioSig event code, e.g. `768`=trial start, `769..772`=class cues, `1023`=rejected) and `onset`/`duration` converted from seconds to samples via the file's own sampling rate.
- `artifacts`: not read by `MotorImageryDataset`'s trial-extraction code path, so this is filled with zeros rather than parsed.

In [ ]:
!pip install -q mne
import mne, numpy as np, os, shutil

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, 'true_labels'), exist_ok=True)

gdf_files = sorted(f for f in os.listdir(RAW_DIR) if f.endswith('.gdf'))
print(f'Found {len(gdf_files)} .gdf files (expect 18: A01T..A09T, A01E..A09E)')

for fname in gdf_files:
    subj_name = fname.replace('.gdf', '')
    raw = mne.io.read_raw_gdf(os.path.join(RAW_DIR, fname), preload=True, verbose=False)
    sfreq = raw.info['sfreq']
    data = raw.get_data()  # (n_channels, n_times), volts, original GDF channel order
    s = (data * 1e6).T     # -> microvolts, (n_times, n_channels)

    onsets_samp = np.round(np.array(raw.annotations.onset) * sfreq).astype(int)
    durations_samp = np.round(np.array(raw.annotations.duration) * sfreq).astype(int)
    etyp = np.array([int(d) for d in raw.annotations.description]).reshape(-1, 1)
    epos = onsets_samp.reshape(-1, 1)
    edur = durations_samp.reshape(-1, 1)
    artifacts = np.zeros((len(etyp), 1))

    np.savez(
        os.path.join(OUT_DIR, subj_name + '.npz'),
        s=s, etyp=etyp, epos=epos, edur=edur, artifacts=artifacts,
    )

    n_trial_starts = int((etyp == 768).sum())
    print(f'{subj_name}: {s.shape[0]} samples, {s.shape[1]} channels, {n_trial_starts} trial-start (768) events')

true_labels_src = os.path.join(RAW_DIR, 'true_labels')
for f in os.listdir(true_labels_src):
    shutil.copy(os.path.join(true_labels_src, f), os.path.join(OUT_DIR, 'true_labels', f))

print('Done. --dst-data-path =', OUT_DIR)

## 4. Run the actual fine-tuning approach

This is `scripts/finetune.sh`'s own configuration (4-class BCI-IV-2a motor imagery, fold 0), just pointed at the Drive-hosted checkpoint/data/results so it's reusable across sessions and directly comparable with the novelty notebook.

In [ ]:
!python src/train_gpt.py \
  --training-style='decoding' \
  --num-decoding-classes=4 \
  --training-steps=10000 \
  --eval_every_n_steps=500 \
  --log-every-n-steps=1000 \
  --num_chunks=2 \
  --per-device-training-batch-size=32 \
  --per-device-validation-batch-size=32 \
  --chunk_len=500 \
  --chunk_ovlp=0 \
  --run-name='actual_baseline_colab' \
  --ft-only-encoder='True' \
  --fold_i=0 \
  --num-encoder-layers=6 \
  --num-hidden-layers=6 \
  --learning-rate=1e-4 \
  --use-encoder='True' \
  --embedding-dim=1024 \
  --pretrained-model='pretrained_model/pytorch_model.bin' \
  --dst-data-path="{OUT_DIR}/" \
  --log-dir="{RESULTS_DIR}"

## 5. Check results

In [ ]:
import pandas as pd, numpy as np, os
RUN_DIR = os.path.join(RESULTS_DIR, 'actual_baseline_colab-0')  # run-name + '-' + fold_i, per train_gpt.py's get_config()
print(pd.read_csv(os.path.join(RUN_DIR, 'test_metrics.csv')))
preds = np.load(os.path.join(RUN_DIR, 'test_predictions.npy'))
labels = np.load(os.path.join(RUN_DIR, 'test_label_ids.npy'))
acc = (preds.argmax(-1) == labels).mean()
print('actual_baseline_colab test accuracy:', acc)